# All-in-One Kanzi Framework 4.1.0 RAG Data Pipeline Notebook

This unified Jupyter Notebook merges **Markdown Text Cleaning** and **VDB Semantic Dataset Building** into a single seamless pipeline.

--- 

## 📌 Pipeline Overview:
### Phase 1: Text Cleaning & Noise Elimination (`clean_kanzi_docs` equivalent)
- Removes mojibake anchor symbols (`Â¶`)
- Cleans Sphinx navigation clutter in frontmatter titles
- Removes local `_images/` tags
- Strips noise headings (`## See also`, `## Related topics`, `## Prerequisites`)
- Converts relative links to plain text
- Formats callout blocks (`> **Tip:** ...`)
- Fixes broken Markdown list items & white space
- Deletes obsolete folders (`licenses/`, `release-notes/kanzi-3.0/`) & stub files (< 200 chars)

### Phase 2: Semantic Chunking & Dataset Export (`build_vdb_dataset` equivalent)
- Splits text along `H1 > H2 > H3` semantic headings
- Classifies content into `prose` vs `code` blocks
- Applies **Contextual Retrieval** headers (`[Document Context: ...]`) [Anthropic Pattern]
- Applies **Parent-Document Retrieval** linking (`parent_id` & `parent_content` ~2,000 chars)
- Performs SHA-256 content deduplication
- Exports standardized `kanzi_rag_chunks.json` for Vector Database ingestion

### 📌 Cell 1: Import Libraries & Configure Environment Paths

In [ ]:
#region Imports & Environment Setup
import os
import re
import shutil
import json
import hashlib
from pathlib import Path

# Set Working Directory & Path Configurations
SCRIPT_DIR = Path.cwd()
DOCS_DIR = (SCRIPT_DIR.parent / "kanzi_docs").resolve()
OUTPUT_JSON = (SCRIPT_DIR.parent / "kanzi_rag_chunks.json").resolve()

# Exclude directories relative to DOCS_DIR
EXCLUDE_DIRS = [
    "licenses",                  # Legal licenses (694 KB)
    "release-notes/kanzi-3.0",  # Obsolete 3.0 release notes (88 files)
]

# Pipeline Sizing Parameters
MIN_BODY_CHARS = 200          # Min body length to keep file (deletes stubs)
TARGET_CHUNK_SIZE = 250       # High-precision small child chunk size (~250 chars) for Vector Search
PARENT_MAX_CHARS = 2000       # Maximum Parent Context size for LLM Prompt generation
MIN_CHUNK_CHARS = 40          # Ignore tiny trailing fragments

print(f"[Config] Working Directory : {SCRIPT_DIR}")
print(f"[Config] Target Docs Dir    : {DOCS_DIR}")
print(f"[Config] Output JSON File   : {OUTPUT_JSON}")
#endregion

### 📌 Cell 2: Phase 1 — Markdown Text Cleaner Functions (11 Rules & Exclusions)
Defines all text transformation rules: Pilcrow removal, frontmatter title fix, link inlining, callout formatting, and noise section stripping.

In [ ]:
#region Phase 1 - Markdown Text Cleaning Rules Engine

_FM_NAV_JUNK = re.compile(r"\s*[-–]?\s*Kanzi framework[\d\s.]*documentation.*$", re.IGNORECASE)
_FM_NAV_EXTRAS = re.compile(r"(ContentsMenuExpand|Light mode|Dark mode|Auto light/dark[^\n]*)")
_IMAGE_MD = re.compile(r"!\[[^\]]*\]\([^)]*_images/[^)]+\)")
_NOISE_SECTION_HEADINGS = re.compile(r"^#{1,6}\s+(See also|Prerequisites|Related topics|In this section|On this page|Contents|Navigation|Next steps?)\s*$", re.IGNORECASE | re.MULTILINE)
_REL_LINK = re.compile(r"\[([^\]]+)\]\((?!https?://)([^)]+\.html[^)]*)\)")
_URL_DUP = re.compile(r"\[(https?://[^\]]+)\]\(\1\)")
_CALLOUT_HEADING = re.compile(r"^(#{1,6})\s+(Tip|Note|Warning|Important|Caution)\s*$", re.IGNORECASE)
_BROKEN_LIST_ITEM = re.compile(r"^-\s*$\n\n(?=\S)", re.MULTILINE)

def clean_markdown_text(text: str) -> str:
    """Applies 11 sequential cleaning rules to raw Markdown text."""
    # 1. Strip Â¶ pilcrow mojibake
    text = text.replace("Â¶", "").replace("¶", "")
    
    # 2. Clean frontmatter title
    lines = text.split("\n")
    fm_clean = []
    in_fm, fm_count = False, 0
    for line in lines:
        if line.rstrip() == "---":
            fm_count += 1
            in_fm = (fm_count == 1)
            fm_clean.append(line)
            continue
        if in_fm and line.startswith("title:"):
            title = line[len("title:"):].strip()
            title = _FM_NAV_JUNK.sub("", title).strip()
            title = _FM_NAV_EXTRAS.sub("", title).strip().rstrip(" -").strip()
            fm_clean.append(f"title: {title}")
        else:
            fm_clean.append(line)
    text = "\n".join(fm_clean)

    # 3. Remove local _images/ tags
    text = _IMAGE_MD.sub("", text)
    
    # 4. Remove noise sections (See also, Related topics, etc.)
    lines = text.split("\n")
    res_lines = []
    skip_level = None
    for line in lines:
        hm = re.match(r"^(#{1,6})\s+(.*)", line)
        if hm:
            level = len(hm.group(1))
            if skip_level is not None:
                if level <= skip_level:
                    skip_level = None
                else:
                    continue
            if _NOISE_SECTION_HEADINGS.match(line):
                skip_level = level
                continue
        elif skip_level is not None:
            continue
        res_lines.append(line)
    text = "\n".join(res_lines)

    # 5. Inline relative HTML links
    text = _REL_LINK.sub(r"\1", text)
    
    # 6. Deduplicate bare URL links
    text = _URL_DUP.sub(r"\1", text)
    
    # 7. Format callout blocks (> **Tip:**)
    lines = text.split("\n")
    co_lines = []
    for line in lines:
        m = _CALLOUT_HEADING.match(line)
        if m:
            kind = m.group(2).capitalize()
            co_lines.append(f"> **{kind}:**")
        else:
            co_lines.append(line)
    text = "\n".join(co_lines)

    # 8. Fix broken list items
    text = _BROKEN_LIST_ITEM.sub("- ", text)
    
    # 8b. Strip blockquote noise (> and empty > lines)
    text = re.sub(r"^\s*>\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*>\s?", "", text, flags=re.MULTILINE)
    # 9. Clean trailing whitespace & 10. collapse blank lines
    lines = [l.rstrip() for l in text.split("\n")]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    
    return text.strip() + "\n"
#endregion

### 📌 Cell 3: Phase 1 Execution — Clean Markdown Source Files
Deletes exclude directories (`licenses/`, `release-notes/kanzi-3.0/`), cleans all `.md` files, and purges short stub files (< 200 body chars).

In [ ]:
#region Run Phase 1 Text Cleaner Pipeline
print("=== Starting Phase 1: Text Cleaning & Noise Elimination ===")

# 1. Delete excluded directories
for rel_dir in EXCLUDE_DIRS:
    target_path = DOCS_DIR / rel_dir
    if target_path.exists():
        shutil.rmtree(target_path)
        print(f"  ok Deleted noise directory: {rel_dir}")

# 2. Scan and clean all Markdown files
all_md_files = sorted(f for f in DOCS_DIR.rglob("*.md") if f.suffix == ".md")
cleaned_count = 0
purged_stubs = 0

for md_file in all_md_files:
    raw_text = md_file.read_text(encoding="utf-8", errors="replace")
    cleaned_text = clean_markdown_text(raw_text)
    
    # Check body character count to remove stubs
    body_only = cleaned_text
    if cleaned_text.startswith("---") and "---" in cleaned_text[3:]:
        body_only = cleaned_text.split("---", 2)[-1].strip()
    
    if len(body_only) < MIN_BODY_CHARS:
        md_file.unlink()
        purged_stubs += 1
    else:
        md_file.write_text(cleaned_text, encoding="utf-8")
        cleaned_count += 1

print(f"\nPhase 1 Complete:")
print(f"  - Cleaned Active Markdown Files: {cleaned_count}")
print(f"  - Purged Stub/Short Files    : {purged_stubs}")
#endregion

### 📌 Cell 4: Phase 2 — VDB Semantic Chunking Engine (Parent-Child & Contextual Prefix)
Parses document structure, applies H2/H3 sectioning, constructs Contextual Prefix headers (`[Document Context: ...]`), and links small child chunks to parent section text (`parent_content`).

In [ ]:
#region Phase 2 - Semantic Chunking Engine
def parse_frontmatter(text: str) -> tuple[dict, str]:
    """Extracts YAML frontmatter dictionary and body string."""
    metadata = {}
    body = text
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) >= 3:
            fm_text = parts[1]
            body = parts[2].strip()
            for line in fm_text.split("\n"):
                if ":" in line:
                    key, val = line.split(":", 1)
                    metadata[key.strip()] = val.strip()
    return metadata, body

def process_file(filepath: Path, docs_root: Path, seen_hashes: set) -> list[dict]:
    """Processes a clean Markdown file into parent-child chunks with Contextual Headers."""
    text = filepath.read_text(encoding="utf-8", errors="replace")
    fm, body = parse_frontmatter(text)

    rel_path = filepath.relative_to(docs_root).as_posix()
    page_title = fm.get("title", filepath.stem)
    source_url = fm.get("source", f"https://docs.kanzi.com/4.1.0/en/{rel_path}")

    heading_pattern = re.compile(r"^(#{1,3})\s+(.+)$", re.MULTILINE)
    matches = list(heading_pattern.finditer(body))
    
    sections = []
    if not matches:
        sections.append({"h_level": 1, "heading": page_title, "content": body})
    else:
        if matches[0].start() > 0:
            pre_text = body[:matches[0].start()].strip()
            if pre_text:
                sections.append({"h_level": 1, "heading": page_title, "content": pre_text})
        
        current_h1, current_h2, current_h3 = page_title, "", ""
        for idx, match in enumerate(matches):
            level = len(match.group(1))
            heading_text = match.group(2).strip()
            start_pos = match.end()
            end_pos = matches[idx + 1].start() if idx + 1 < len(matches) else len(body)
            section_content = body[start_pos:end_pos].strip()

            if level == 1:
                current_h1, current_h2, current_h3 = heading_text, "", ""
            elif level == 2:
                current_h2, current_h3 = heading_text, ""
            elif level == 3:
                current_h3 = heading_text

            path_parts = [p for p in [current_h1, current_h2, current_h3] if p]
            section_path = " > ".join(path_parts)
            if section_content:
                sections.append({"h_level": level, "heading": heading_text, "section_path": section_path, "content": section_content})

    chunks = []
    for sec_idx, sec in enumerate(sections, start=1):
        section_path = sec.get("section_path", page_title)
        sec_content = sec["content"]
        
        parent_id = f"{rel_path}#sec-{sec_idx}"
        ctx_prefix = f"[Document Context: {section_path}]" if section_path else f"[Document Context: {page_title}]"
        parent_content = f"{ctx_prefix}\n\n{sec_content}"

        code_block_pattern = re.compile(r"```(\w*)\n(.*?)```", re.DOTALL)
        last_end = 0
        for cb_match in code_block_pattern.finditer(sec_content):
            cb_start, cb_end = cb_match.span()
            prose_part = sec_content[last_end:cb_start].strip()
            code_lang = cb_match.group(1) or "text"
            code_content = cb_match.group(2).strip()

            if len(prose_part) >= MIN_CHUNK_CHARS:
                chunks.extend(_subchunk_prose(prose_part, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes))

            if len(code_content) >= 15:
                code_text = f"{ctx_prefix}\n```{code_lang}\n{code_content}\n```"
                c_hash = hashlib.sha256(code_text.encode("utf-8")).hexdigest()
                if c_hash not in seen_hashes:
                    seen_hashes.add(c_hash)
                    chunks.append({
                        "id": f"{rel_path}#code-{len(chunks)+1}",
                        "content": code_text,
                        "chunk_type": "code",
                        "code_lang": code_lang,
                        "metadata": {
                            "source_url": source_url,
                            "local_path": rel_path,
                            "page_title": page_title,
                            "section_path": section_path,
                            "chunk_type": "code",
                            "code_lang": code_lang,
                            "parent_id": parent_id,
                            "parent_content": parent_content[:PARENT_MAX_CHARS],
                            "context_enriched": True,
                            "char_count": len(code_text),
                            "hash": c_hash
                        }
                    })
            last_end = cb_end

        remaining_prose = sec_content[last_end:].strip()
        if len(remaining_prose) >= MIN_CHUNK_CHARS:
            chunks.extend(_subchunk_prose(remaining_prose, section_path, page_title, source_url, rel_path, parent_id, parent_content, seen_hashes))

    return chunks

def _subchunk_prose(text: str, section_path: str, page_title: str, source_url: str, rel_path: str, parent_id: str, parent_content: str, seen_hashes: set) -> list[dict]:
    """Helper splitting prose into ~250 char child chunks linked to Parent Content."""
    prose_chunks = []
    if len(text) <= TARGET_CHUNK_SIZE + 100:
        sub_texts = [text]
    else:
        paragraphs = text.split("\n\n")
        sub_texts = []
        curr_buf = ""
        for p in paragraphs:
            p = p.strip()
            if not p:
                continue
            if len(curr_buf) + len(p) <= TARGET_CHUNK_SIZE:
                curr_buf = f"{curr_buf}\n\n{p}" if curr_buf else p
            else:
                if curr_buf:
                    sub_texts.append(curr_buf)
                curr_buf = p
        if curr_buf:
            sub_texts.append(curr_buf)

    for st in sub_texts:
        st_clean = st.strip()
        if len(st_clean) < MIN_CHUNK_CHARS:
            continue

        ctx_prefix = f"[Document Context: {section_path}]" if section_path else f"[Document Context: {page_title}]"
        enriched_content = f"{ctx_prefix}\n\n{st_clean}"
        c_hash = hashlib.sha256(enriched_content.encode("utf-8")).hexdigest()
        if c_hash in seen_hashes:
            continue
        seen_hashes.add(c_hash)

        prose_chunks.append({
            "id": f"{rel_path}#{c_hash[:8]}",
            "content": enriched_content,
            "chunk_type": "prose",
            "code_lang": None,
            "metadata": {
                "source_url": source_url,
                "local_path": rel_path,
                "page_title": page_title,
                "section_path": section_path,
                "chunk_type": "prose",
                "code_lang": "none",
                "parent_id": parent_id,
                "parent_content": parent_content[:PARENT_MAX_CHARS],
                "context_enriched": True,
                "char_count": len(enriched_content),
                "hash": c_hash
            }
        })
    return prose_chunks
#endregion

### 📌 Cell 5: Phase 2 Execution — Build & Export Dataset Payload
Executes chunking across clean Markdown files and writes `kanzi_rag_chunks.json`.

In [ ]:
#region Run Phase 2 Dataset Builder & Export JSON
print("=== Starting Phase 2: Semantic Chunking & Dataset Building ===")
md_files = sorted(f for f in DOCS_DIR.rglob("*.md") if f.suffix == ".md")
print(f"  ok Processing {len(md_files)} clean Markdown source files...")

all_chunks = []
seen_hashes = set()
code_count, prose_count = 0, 0

for filepath in md_files:
    file_chunks = process_file(filepath, DOCS_DIR, seen_hashes)
    for c in file_chunks:
        if c["chunk_type"] == "code":
            code_count += 1
        else:
            prose_count += 1
    all_chunks.extend(file_chunks)

dataset_payload = {
    "dataset_name": "kanzi-framework-4.1.0-rag-chunks",
    "doc_version": "4.1.0",
    "total_source_files": len(md_files),
    "total_chunks": len(all_chunks),
    "prose_chunks_count": prose_count,
    "code_chunks_count": code_count,
    "chunks": all_chunks
}

OUTPUT_JSON.write_text(json.dumps(dataset_payload, indent=2, ensure_ascii=False), encoding="utf-8")
size_mb = OUTPUT_JSON.stat().st_size / (1024 * 1024)

print(f"\nPhase 2 Complete:")
print(f"  - Total Chunks Exported : {len(all_chunks)}")
print(f"  - Prose Chunks          : {prose_count}")
print(f"  - Code Chunks           : {code_count}")
print(f"  - Output JSON Payload   : {OUTPUT_JSON.name} ({size_mb:.2f} MB)")
#endregion

### 📌 Cell 6: Phase 3 — Verification & Inspection
Loads the exported JSON dataset to inspect Child Search Content vs Parent LLM Context.

In [ ]:
#region Phase 3 Verification & Inspection
with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    verify_data = json.load(f)

print(f"Loaded Dataset Name : {verify_data['dataset_name']}")
print(f"Total Vector Chunks : {verify_data['total_chunks']}")

sample_chunk = verify_data["chunks"][0]
print(f"\nSample Chunk Inspection (Chunk #1):")
print(f"  [Chunk ID]      : {sample_chunk['id']}")
print(f"  [Search Content]: {sample_chunk['content'][:140]}...")
print(f"  [Parent ID]     : {sample_chunk['metadata']['parent_id']}")
print(f"  [Parent Context]: {sample_chunk['metadata']['parent_content'][:180]}...")
#endregion